# Procedural noise, one construction at a time — experiments

Companion notebook to the post. The notebook treats each method as a construction rather than as a visual label, then studies the operators that are commonly placed on top of those constructions.

The experiments cover:

1. value, Perlin, simplex, and OpenSimplex base fields;
2. one-dimensional cuts and finite-difference derivatives;
3. angular power spectra as a diagnostic for lattice bias;
4. cellular features $F_1$, $F_2$, and $F_2-F_1$ under three metrics;
5. Gabor orientation, phasor profiles, global wave sums, and wavelet tiles;
6. fBm, ridged, and billow octave stacks;
7. domain warping and displacement statistics;
8. spatial fields paired with log power spectra;
9. a JSON manifest with parameters and basic field statistics.

The code uses the actual `biomeforge.noise` implementations. The browser demos in the post are deliberately smaller, live versions of the same mechanisms.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize, TwoSlopeNorm

# --- locate BiomeForge -----------------------------------------------------
def import_noise_module():
    try:
        import biomeforge.noise as noise
        return noise
    except ImportError:
        pass

    candidates = []
    env_root = os.environ.get("BIOMEFORGE_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "biomeforge").is_dir():
            sys.path.insert(0, str(candidate))
            import biomeforge.noise as noise
            return noise

    raise ImportError(
        "Could not import biomeforge.noise. Install BiomeForge, run from its "
        "project tree, or set BIOMEFORGE_ROOT=/path/to/project."
    )

N = import_noise_module()

# --- output directory -----------------------------------------------------
post_artifacts = Path("posts/a-small-taxonomy-of-procedural-noise/artifacts")
if post_artifacts.is_dir():
    ART = post_artifacts
elif Path.cwd().name == "artifacts":
    ART = Path.cwd()
else:
    ART = Path.cwd() / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

# --- site theme -----------------------------------------------------------
BG, PANEL, LINE = "#0e1116", "#151a21", "#222a35"
FG, MUTED, ACCENT, ACCENT2 = "#d7dde6", "#8a94a3", "#4aa3ff", "#e0a458"

mpl.rcParams.update({
    "figure.facecolor": PANEL,
    "axes.facecolor": PANEL,
    "savefig.facecolor": PANEL,
    "axes.edgecolor": LINE,
    "axes.labelcolor": FG,
    "text.color": FG,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": LINE,
    "axes.grid": False,
    "font.family": "monospace",
    "font.size": 10,
    "figure.dpi": 120,
    "axes.titlesize": 11,
})

SEED = 17
SIZE = 256
EXTENT = 8.0
axis = np.linspace(0.0, EXTENT, SIZE, endpoint=False)
X, Y = np.meshgrid(axis, axis)

print(f"BiomeForge noises: {', '.join(N.list_noises())}")
print(f"grid={SIZE}x{SIZE}, extent=[0,{EXTENT}), seed={SEED}, output={ART}")

In [ ]:
@dataclass(frozen=True)
class Panel:
    name: str
    field: np.ndarray
    family: str
    description: str
    signed: bool = True


def finite_field(field):
    arr = np.asarray(field, dtype=np.float64)
    if arr.ndim != 2:
        raise ValueError(f"expected a 2-D field, got {arr.shape}")
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def stats(field):
    arr = finite_field(field)
    return {
        "min": float(arr.min()),
        "max": float(arr.max()),
        "mean": float(arr.mean()),
        "std": float(arr.std()),
        "rms": float(np.sqrt(np.mean(arr * arr))),
    }


def display_norm(field, signed=True):
    arr = finite_field(field)
    if signed:
        limit = max(float(np.quantile(np.abs(arr), 0.995)), 1e-6)
        return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit), "coolwarm"
    lo, hi = np.quantile(arr, [0.005, 0.995])
    if hi <= lo:
        hi = lo + 1e-6
    return Normalize(vmin=float(lo), vmax=float(hi)), "viridis"


def save_grid(panels, filename, title, columns=4):
    rows = math.ceil(len(panels) / columns)
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(4.25 * columns, 4.0 * rows),
        constrained_layout=True,
        squeeze=False,
    )
    for ax, panel in zip(axes.ravel(), panels):
        field = finite_field(panel.field)
        norm, cmap = display_norm(field, panel.signed)
        image = ax.imshow(
            field,
            origin="lower",
            extent=(0.0, EXTENT, 0.0, EXTENT),
            cmap=cmap,
            norm=norm,
            interpolation="bilinear",
        )
        st = stats(field)
        ax.set_title(panel.name)
        ax.set_xlabel(f"mean={st['mean']:.3f}, std={st['std']:.3f}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(image, ax=ax, shrink=0.70, pad=0.02)
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=16)
    path = ART / filename
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)
    print(path)
    return path


def power_spectrum(field):
    arr = finite_field(field)
    arr = arr - arr.mean()
    window = np.outer(np.hanning(arr.shape[0]), np.hanning(arr.shape[1]))
    fft = np.fft.fftshift(np.fft.fft2(arr * window))
    return np.log10(np.abs(fft) ** 2 + 1e-12)

## 1. Base constructions

The first experiment compares base methods before any octave stacking or coordinate warping.

- **Value noise:** scalar coefficients on a lattice.
- **Perlin noise:** gradient coefficients on a Cartesian lattice.
- **Simplex noise:** compact gradient contributions on a simplicial lattice.
- **OpenSimplex:** gradient contributions on a stretched/squished alternative lattice.
- **Cellular/Worley:** feature-point positions; the natural output is a distance.
- **Gabor:** localized oscillatory impulses.
- **Phasor:** a coherent stochastic phase passed through a periodic profile.
- **Wave:** global plane-wave components.
- **Wavelet:** coefficients in a periodic band-limited tile.

The gallery is intentionally descriptive rather than normative: the methods solve different problems.


In [ ]:
base_panels = [
    Panel("Value", N.value2(X, Y, seed=SEED), "base", "Interpolated random lattice values."),
    Panel("Perlin", N.perlin2(X, Y, seed=SEED), "base", "Gradient noise on a Cartesian lattice."),
    Panel("Simplex", N.simplex2(X, Y, seed=SEED), "base", "Gradient noise on a simplex lattice."),
    Panel("OpenSimplex", N.opensimplex2(X, Y, seed=SEED), "base", "Alternative stretched gradient lattice."),
    Panel("Cellular F1", N.cellular2(X, Y, seed=SEED, feature="F1"), "base",
          "Distance to nearest feature point.", signed=False),
    Panel("Cellular F2-F1", N.cellular2(X, Y, seed=SEED, feature="F2F1"), "base",
          "Nearest/second-nearest boundary signal.", signed=False),
    Panel("Gabor isotropic", N.gabor2(X, Y, seed=SEED, frequency=0.9, orientation=None),
          "base", "Localized oscillatory kernels with random directions."),
    Panel("Gabor oriented", N.gabor2(X, Y, seed=SEED, frequency=0.9, orientation=0.45),
          "base", "Localized oscillatory kernels with a common direction."),
    Panel("Phasor", N.phasor2(X, Y, seed=SEED, frequency=0.9, orientation=0.4, profile="sin"),
          "base", "Stochastic phase passed through a sinusoidal profile."),
    Panel("Wave", N.wave2(X, Y, seed=SEED, frequency=0.7, profile="cos"),
          "base", "Random global plane-wave interference."),
    Panel("Wavelet", N.wavelet2(X * 5.0, Y * 5.0, seed=SEED),
          "base", "Band-limited periodic wavelet tile."),
]

save_grid(base_panels, "noise-zoo-base.png", "Noise zoo: base families")

### 1.1 Cross-sections and finite-difference derivatives

Two images can look similarly smooth while their local derivatives behave differently. The following experiment takes the same horizontal cut through the four lattice/gradient methods and estimates $\partial N/\partial x$ with a centered finite difference.

The derivative plot is not a proof of regularity; it is a practical diagnostic for cell-boundary structure and scale-dependent roughness.

In [ ]:
cut_x = np.linspace(-5.0, 5.0, 2400)
cut_y = np.full_like(cut_x, 0.371)
cut_methods = [
    ("Value", N.value2),
    ("Perlin", N.perlin2),
    ("Simplex", N.simplex2),
    ("OpenSimplex", N.opensimplex2),
]

fig, axes = plt.subplots(2, 1, figsize=(10.2, 6.8), sharex=True)
for name, fn in cut_methods:
    field = np.asarray(fn(cut_x, cut_y, seed=SEED), dtype=float)
    deriv = np.gradient(field, cut_x)
    axes[0].plot(cut_x, field, lw=1.25, label=name)
    axes[1].plot(cut_x, deriv, lw=1.0, label=name)

axes[0].set_ylabel("field")
axes[0].set_title("A common one-dimensional cut")
axes[1].set_ylabel("finite-difference derivative")
axes[1].set_xlabel("x")
axes[1].set_title("Local slope along the same cut")
for ax in axes:
    style(ax)
    ax.legend(ncol=4, fontsize=9)
fig.tight_layout()
fig.savefig(ART / "base-cross-sections.png", bbox_inches="tight")
plt.show()


### 1.2 Angular power as a lattice-bias diagnostic

A lattice can leave preferred directions even when the spatial image looks organic. We window each field, compute its power spectrum, and bin spectral energy by angle. A perfectly isotropic field would give a flat angular curve after radial aggregation; peaks indicate preferred orientations.

In [ ]:
def angular_power(field, bins=72):
    arr = finite_field(field)
    wy = np.hanning(arr.shape[0])[:, None]
    wx = np.hanning(arr.shape[1])[None, :]
    F = np.fft.fftshift(np.fft.fft2((arr - arr.mean()) * wy * wx))
    P = np.abs(F) ** 2
    yy, xx = np.indices(P.shape)
    cy = (P.shape[0] - 1) / 2
    cx = (P.shape[1] - 1) / 2
    theta = np.mod(np.arctan2(yy - cy, xx - cx), np.pi)
    radius = np.hypot(xx - cx, yy - cy)
    mask = radius > 4
    edges = np.linspace(0, np.pi, bins + 1)
    out = np.zeros(bins)
    for k in range(bins):
        m = mask & (theta >= edges[k]) & (theta < edges[k + 1])
        out[k] = P[m].mean() if np.any(m) else 0.0
    out /= out.mean() + 1e-12
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, out

fig, ax = plt.subplots(figsize=(8.6, 4.5))
for name, fn in cut_methods:
    field = fn(X, Y, seed=SEED)
    angle, power = angular_power(field)
    ax.plot(np.degrees(angle), power, lw=1.25, label=name)
ax.axhline(1.0, color=MUTED, ls="--", lw=1.0)
ax.set(xlabel="orientation modulo 180 degrees", ylabel="relative angular power",
       title="Preferred spectral directions")
style(ax)
ax.legend(ncol=4, fontsize=9)
fig.tight_layout()
fig.savefig(ART / "base-angular-power.png", bbox_inches="tight")
plt.show()


## 2. Octave combinators

An octave stack evaluates one base field at increasing frequencies and decreasing amplitudes,

\[
F(x)=\frac{1}{Z}\sum_{o=0}^{m-1}a_oN(f_ox;s_o),
\qquad f_o=f_0L^o,\quad a_o=a_0G^o.
\]

The three variants below differ only in the per-octave remap:

\[
\text{fBm}: n,\qquad
\text{ridged}: (1-|n|)^2,\qquad
\text{billow}: 2|n|-1.
\]

These are compositions over a base noise, not independent base constructions.

In [ ]:
bases: Sequence[tuple[str, Callable]] = [
    ("Value", N.value2),
    ("Perlin", N.perlin2),
    ("Simplex", N.simplex2),
    ("OpenSimplex", N.opensimplex2),
]
kinds = [("fBm", "fbm"), ("Ridged", "ridged"), ("Billow", "billow")]

octave_panels = []
for base_name, base_fn in bases:
    for display_name, kind in kinds:
        field = N.fractal2(
            base_fn,
            X,
            Y,
            seed=SEED,
            octaves=6,
            lacunarity=2.0,
            gain=0.5,
            kind=kind,
        )
        octave_panels.append(
            Panel(f"{base_name} {display_name}", field, "octaves", f"{kind} octave stack.")
        )

save_grid(octave_panels, "noise-zoo-octaves.png", "Noise zoo: octave combinators")

## 3. Cellular features and metrics

For feature sites \(p_i\), let \(F_k(x)\) be the distance to the \(k\)-th nearest site.

- \(F_1\) describes cell interiors, pits, and spots.
- \(F_2\) is the second-nearest distance.
- \(F_2-F_1\) approaches zero where two sites compete, so it exposes cell walls.

The metric is part of the construction. Euclidean, Manhattan, and Chebyshev distance produce different cell geometry even when the feature sites are unchanged.

In [ ]:
cellular_panels = []
for metric in ("euclidean", "manhattan", "chebyshev"):
    for feature in ("F1", "F2", "F2F1"):
        cellular_panels.append(
            Panel(
                f"{metric.title()} {feature}",
                N.cellular2(X * 0.9, Y * 0.9, seed=SEED, feature=feature, metric=metric),
                "cellular",
                f"{feature} with {metric} distance.",
                signed=False,
            )
        )

save_grid(
    cellular_panels,
    "noise-zoo-cellular.png",
    "Noise zoo: cellular features and metrics",
)

## 4. Spectral and periodic families

Gabor noise gives local control of frequency, bandwidth, and orientation. Phasor noise separates the phase field from the one-dimensional periodic profile. Wave noise samples global plane-wave components. Wavelet noise reconstructs a periodic band-limited tile.

The plots below isolate those controls: Gabor orientation, phasor profile, wave profile, and wavelet periodicity.

In [ ]:
spectral_panels = [
    Panel("Gabor isotropic",
          N.gabor2(X, Y, seed=SEED, frequency=0.8, bandwidth=1.5, orientation=None),
          "spectral", "Random kernel orientations."),
    Panel("Gabor 0 degrees",
          N.gabor2(X, Y, seed=SEED, frequency=0.8, bandwidth=1.5, orientation=0.0),
          "spectral", "Common orientation at 0 degrees."),
    Panel("Gabor 45 degrees",
          N.gabor2(X, Y, seed=SEED, frequency=0.8, bandwidth=1.5, orientation=math.pi / 4),
          "spectral", "Common orientation at 45 degrees."),
    Panel("Gabor 90 degrees",
          N.gabor2(X, Y, seed=SEED, frequency=0.8, bandwidth=1.5, orientation=math.pi / 2),
          "spectral", "Common orientation at 90 degrees."),
    Panel("Phasor sine",
          N.phasor2(X, Y, seed=SEED, frequency=0.8, orientation=0.35, profile="sin"),
          "spectral", "Smooth periodic profile."),
    Panel("Phasor triangle",
          N.phasor2(X, Y, seed=SEED, frequency=0.8, orientation=0.35, profile="triangle"),
          "spectral", "Piecewise-linear periodic profile."),
    Panel("Phasor sawtooth",
          N.phasor2(X, Y, seed=SEED, frequency=0.8, orientation=0.35, profile="sawtooth"),
          "spectral", "Directional ramp profile."),
    Panel("Phasor square",
          N.phasor2(X, Y, seed=SEED, frequency=0.8, orientation=0.35, profile="square"),
          "spectral", "Binary periodic profile."),
    Panel("Wave cosine",
          N.wave2(X, Y, seed=SEED, frequency=0.65, bandwidth=0.55, profile="cos"),
          "spectral", "Gaussian-like random wave sum."),
    Panel("Wave sine phase",
          N.wave2(X, Y, seed=SEED, frequency=0.65, bandwidth=0.55, profile="sin"),
          "spectral", "Complex wave phase through sine."),
    Panel("Wave sawtooth phase",
          N.wave2(X, Y, seed=SEED, frequency=0.65, bandwidth=0.55, profile="sawtooth"),
          "spectral", "Complex wave phase through sawtooth."),
    Panel("Wavelet tile",
          N.wavelet2(X * 5.0, Y * 5.0, seed=SEED, tile_size=48),
          "spectral", "Periodic band-pass noise."),
]

save_grid(
    spectral_panels,
    "noise-zoo-spectral.png",
    "Noise zoo: spectral and periodic families",
)

### 4.1 The phase field and the profile are separate objects

For phasor-style methods, the spatial organization is carried by a phase $\Phi(x)$. The chosen periodic profile only maps one phase cycle to an output value. Plotting those profiles directly makes clear which changes preserve smoothness and which introduce discontinuities.

In [ ]:
phase = np.linspace(-np.pi, np.pi, 1400)
p = (phase / (2 * np.pi)) % 1.0
profiles = {
    "sine": np.sin(phase),
    "triangle": 2.0 * np.abs(2.0 * p - 1.0) - 1.0,
    "sawtooth": 2.0 * p - 1.0,
    "square": np.where(p < 0.5, 1.0, -1.0),
}
fig, ax = plt.subplots(figsize=(9.2, 4.3))
for name, values in profiles.items():
    ax.plot(phase / np.pi, values, lw=1.35, label=name)
ax.set(xlabel="phase / pi", ylabel="profile value", title="One phase cycle, four output profiles")
style(ax)
ax.legend(ncol=4)
fig.tight_layout()
fig.savefig(ART / "phasor-profiles.png", bbox_inches="tight")
plt.show()


## 5. Domain warping

A domain warp composes a base field with a displacement map,

\[
\phi(x)=x+Aq(x),\qquad Y(x)=N(\phi(x)).
\]

The base generator controls the local statistics. The displacement field controls how the coordinate system bends. The difference image makes the geometric effect explicit.

In [ ]:
warp_panels = []
for base_name, base_fn in bases:
    plain = finite_field(base_fn(X, Y, seed=SEED))
    warped = finite_field(
        N.warp2(base_fn, X, Y, seed=SEED, amp=1.25, freq=1.0, octaves=4)
    )
    warp_panels.extend([
        Panel(f"{base_name} plain", plain, "warp", "Unwarped base field."),
        Panel(f"{base_name} warped", warped, "warp", "Coordinates displaced by two fBm fields."),
        Panel(f"{base_name} difference", warped - plain, "warp", "Warped minus unwarped."),
    ])

save_grid(warp_panels, "noise-zoo-warp.png", "Noise zoo: domain warping")

### 5.1 Inspect the displacement map, not only the warped image

A warp is a vector field. Its amplitude distribution and finite-difference Jacobian indicate how strongly coordinates are translated, compressed, or stretched. The following diagnostic uses the OpenSimplex basis and the same two-offset construction as `warp_coords2`.

In [ ]:
warp_base = N.opensimplex2
warp_amp = 1.25
warp_freq = 1.0
warp_octaves = 4

qx = N.fractal2(warp_base, X * warp_freq, Y * warp_freq,
                seed=SEED + 31, octaves=warp_octaves)
qy = N.fractal2(warp_base, X * warp_freq + 5.2, Y * warp_freq + 1.3,
                seed=SEED + 37, octaves=warp_octaves)
displacement = warp_amp * np.hypot(qx, qy)

dx = float(X[0, 1] - X[0, 0])
dy = float(Y[1, 0] - Y[0, 0])
dqx_dy, dqx_dx = np.gradient(qx, dy, dx)
dqy_dy, dqy_dx = np.gradient(qy, dy, dx)
# Frobenius norm of J_phi - I = A * J_q.
jacobian_deviation = warp_amp * np.sqrt(
    dqx_dx**2 + dqx_dy**2 + dqy_dx**2 + dqy_dy**2
)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for ax, field, title in [
    (axes[0], displacement, "displacement magnitude"),
    (axes[1], jacobian_deviation, "local Jacobian deviation"),
]:
    im = ax.imshow(field, origin="lower", cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(ART / "warp-displacement-diagnostics.png", bbox_inches="tight")
plt.show()

print(f"mean displacement: {displacement.mean():.3f}")
print(f"95% displacement: {np.quantile(displacement, .95):.3f}")
print(f"95% Jacobian deviation: {np.quantile(jacobian_deviation, .95):.3f}")


## 6. Spatial fields and power spectra

Visual similarity can hide spectral differences. The log power spectrum shows whether energy is isotropic, axis-biased, concentrated around a preferred frequency, or band-limited.

A Hann window is applied before the FFT to reduce leakage from non-periodic image boundaries.

In [ ]:
selected = [
    base_panels[0],  # value
    base_panels[1],  # Perlin
    base_panels[2],  # simplex
    base_panels[3],  # OpenSimplex
    spectral_panels[2],  # oriented Gabor
    spectral_panels[4],  # phasor
    spectral_panels[8],  # wave
    spectral_panels[11], # wavelet
]

fig, axes = plt.subplots(
    len(selected), 2, figsize=(9.0, 3.65 * len(selected)),
    constrained_layout=True, squeeze=False
)
for row, panel in enumerate(selected):
    field = finite_field(panel.field)
    norm, cmap = display_norm(field, True)
    axes[row, 0].imshow(field, origin="lower", cmap=cmap, norm=norm, interpolation="bilinear")
    axes[row, 0].set_title(f"{panel.name}: spatial field")
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])

    axes[row, 1].imshow(power_spectrum(field), origin="lower", cmap="magma")
    axes[row, 1].set_title(f"{panel.name}: log power spectrum")
    axes[row, 1].set_xticks([])
    axes[row, 1].set_yticks([])

fig.suptitle("Noise zoo: spatial fields and spectra", fontsize=16)
spectrum_path = ART / "noise-zoo-spectra.png"
fig.savefig(spectrum_path, dpi=150)
plt.show()
plt.close(fig)
print(spectrum_path)

## 7. Basic statistics and manifest

Mean and standard deviation are not a full characterization, but they are useful checks for accidental bias, clipping, or scaling differences. The manifest records the parameters and statistics used for the galleries.

In [ ]:
pages = {
    "base": base_panels,
    "octaves": octave_panels,
    "cellular": cellular_panels,
    "spectral": spectral_panels,
    "warp": warp_panels,
}

manifest = {
    "seed": SEED,
    "size": SIZE,
    "extent": EXTENT,
    "outputs": {
        "base": "noise-zoo-base.png",
        "octaves": "noise-zoo-octaves.png",
        "cellular": "noise-zoo-cellular.png",
        "spectral": "noise-zoo-spectral.png",
        "warp": "noise-zoo-warp.png",
        "spectra": "noise-zoo-spectra.png",
    },
    "fields": [],
}

print(f"{'page':<11} {'field':<28} {'mean':>9} {'std':>9} {'rms':>9}")
print("-" * 70)
for page, panels in pages.items():
    for panel in panels:
        st = stats(panel.field)
        manifest["fields"].append({
            "page": page,
            "name": panel.name,
            "family": panel.family,
            "description": panel.description,
            "signed": panel.signed,
            "statistics": st,
        })
        print(f"{page:<11} {panel.name:<28} {st['mean']:>9.3f} {st['std']:>9.3f} {st['rms']:>9.3f}")

manifest_path = ART / "noise-zoo-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"\n{manifest_path}")

## Selection rule

Start from the required output and support, then inspect the spectrum at the scale where the field will actually be used.

| Requirement | Starting point | Main diagnostic |
|---|---|---|
| Smooth scalar modulation | Perlin, simplex, OpenSimplex | axis bias and derivative behavior |
| Cells, pits, or cracks | Cellular $F_1$, $F_2-F_1$ | output units and metric geometry |
| Direction and frequency control | Gabor, phasor, wave | angular and radial power spectra |
| Tileable band-limited detail | Wavelet | repetition period and filtering |
| Multiscale roughness | fBm, ridged, billow | highest resolved octave |
| Flowing or folded geometry | Domain warp | displacement and Jacobian magnitude |

Noise supplies structured residual variation. Ownership, connectivity, conservation, and physical causality generally require explicit geometry, graphs, categorical regions, or simulation.
